In [ ]:
import os
import json
import pandas as pd
import re
import numpy as np


# Connect to Google Drive
import gspread
import gspread_dataframe
from google.oauth2.service_account import Credentials
from google.oauth2 import service_account
from googleapiclient.discovery import build
from gspread_dataframe import set_with_dataframe
from gspread_dataframe import get_as_dataframe

In [ ]:
# 1. Fetch credentials from environment variable
creds_env = os.environ.get("GDRIVE_CREDENTIALS_KC")

if not creds_env:
    raise ValueError("Environment variable 'GDRIVE_CREDENTIALS' was not found.")

creds_json = json.loads(creds_env)

# 2. Define required scopes
scopes = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]

# 3. Authenticate service account
creds = service_account.Credentials.from_service_account_info(
    creds_json, scopes=scopes
)

# 4. Initialize Google API clients
drive_service = build("drive", "v3", credentials=creds)
sheets_service = build("sheets", "v4", credentials=creds)

gc = gspread.authorize(creds)

print("Google Drive and Sheets services successfully initialized.")

Mounted at /content/drive


In [4]:
# Open files

# Organic
sprinklr_organic = gc.open_by_key('1FnauIqLuTe1c2N8Z-HQPy8wambQzBhpbLJY24JMCMNY')
sprinklr_organic = sprinklr_organic.worksheet('Hoja 1')
sprinklr_organic = get_as_dataframe(sprinklr_organic)

influencers = gc.open_by_key('1QwqDvUu5SAt6PHKBZWWOATkqzE58pN_XDnzo6LMiYOI')
influencers = influencers.worksheet('Sheet1')
influencers = get_as_dataframe(influencers)

paid_data = gc.open_by_key('1W73RHKRuKfp-AAVQDgrMSwP3huq0r8-bDeRMLjbPxZA')
paid_data = paid_data.worksheet('Hoja 1')
paid_data = get_as_dataframe(paid_data)

tiktok_profile = gc.open_by_key('1947Wx86ZtNWQSaqcYVSXv_3WLvIA0p6u_Ol1DZ8GmX8')
tiktok_profile = tiktok_profile.worksheet('tiktok_profile')
tiktok_profile = get_as_dataframe(tiktok_profile)

instagram_profile = gc.open_by_key('1VK7_oyA3boJaudPaAiwk7xYl6sxReed63eOYBP9ahxo')
instagram_profile = instagram_profile.worksheet('instagram_profile')
instagram_profile = get_as_dataframe(instagram_profile)

internal_organic = gc.open_by_key('1ZPVLBEfQWpVKO-DLxHYUu2xFW1URgpSBwQgHwf6ZcCM')
internal_organic = internal_organic.worksheet('DASHBOARD')
internal_organic = get_as_dataframe(internal_organic)

omd_table = gc.open_by_key('1MNgL9Ugll72xu5z7R8jW7TsvP9-3SyQDlizwqmdrXiY')
omd_table = omd_table.worksheet('Sheet1')
omd_table = get_as_dataframe(omd_table)

In [5]:
# Prepare influencers data
influencers = influencers[
    ['Country', 'brand', 'platform', 'Organic_ID', 'url', 'Accionable', 'run_datetime']
]
influencers['run_datetime'] = pd.to_datetime(influencers['run_datetime']).dt.date
influencers['content_type'] = 'Creator'

In [6]:
# Prepare organic data
sprinklr_organic = sprinklr_organic[
    ['Country of Origin (Account)', 'Brand (Account)', 'Social Network',
     'Organic_ID', 'Permalink (EXTERNAL_VALUE)', 'Accionable', 'Last Updated At']
]
sprinklr_organic.columns = ['Country', 'brand', 'platform', 'Organic_ID', 'url', 'Accionable', 'run_datetime']
sprinklr_organic['run_datetime'] = pd.to_datetime(sprinklr_organic['run_datetime'], format='mixed').dt.date
sprinklr_organic['content_type'] = 'Owned'

In [7]:
# Append organic and influencers data
final_boosting = pd.concat([sprinklr_organic, influencers], ignore_index=True)

In [8]:
# Append TikTok Profile and Instagram profile and prepare for join
combined_profiles = pd.concat([tiktok_profile, instagram_profile], ignore_index=True)

# Extract organic id
def extract_organic_id(link):
    if not isinstance(link, str):
        return None
    url = link.split('?')[0].rstrip('/')
    # TikTok video/photo: .../video/<id> or .../photo/<id>
    m = re.search(r'/(?:video|photo)/(\d+)', url)
    if m:
        return m.group(1)
    # Instagram stories: .../stories/<user>/<id>
    m = re.search(r'/stories/[^/]+/(\d+)', url)
    if m:
        return m.group(1)
    # Instagram reel/reels/p/tv: .../reel|reels|p|tv/<code>
    m = re.search(r'/(?:reel|reels|p|tv)/([A-Za-z0-9_-]+)', url)
    if m:
        return m.group(1)
    # Fallback: last path segment
    return url.rsplit('/', 1)[-1] or None

combined_profiles['Organic_ID'] = combined_profiles['Link of Post'].apply(extract_organic_id)

# Rename columns
combined_profiles = combined_profiles.rename(columns={
    'Date': 'date_sent',
    'Plataform': 'Platform',
    '¿Cuál es la agencia que produjo el contenido?': 'agency',
    'El post puede recibir boost?': 'can_be_boosted',
    'Autorización spark de influenciador': 'spark_auth',
    'Codigo spark para influenciadores': 'spark_code',
    'Link direccionamiento': 'redirectioning_link',
    'Fecha finalización de pauta (fecha)': 'end_date',
    'Objetivo: alcance, interacción, etc': 'goal',
    'Budget: AON / campaña': 'budget',
})

# end_date as date
combined_profiles['end_date'] = pd.to_datetime(combined_profiles['end_date'], format='mixed').dt.date

# Select useful columns
combined_profiles = combined_profiles[[
    'date_sent',
    'agency',
    'can_be_boosted',
    'spark_auth',
    'spark_code',
    'redirectioning_link',
    'end_date',
    'goal',
    'budget',
    'Organic_ID',
]]

# Add preffix
combined_profiles = combined_profiles.add_prefix('prof_')

In [9]:
# Prepare internal_organic for join

# Rename columns
internal_organic = internal_organic.rename(columns={
    'Budget': 'budget',
    'Plataform': 'Platform',
    'Objetivo \n(awareness, consideración, conversión)': 'goal',
    'Territorio de comunicación': 'country',
    'Fecha finalización': 'end_date',
    'Link direccionamiento': 'redirectioning_link',

})

# end_date as date
internal_organic['end_date'] = pd.to_datetime(internal_organic['end_date'], format='mixed').dt.date


# Add preffix
internal_organic = internal_organic.add_prefix('int_')

In [10]:
# Prepare omd_table for join

# Rename columns
omd_table = omd_table.rename(columns={
    'Organic ID': 'Organic_ID'
})

# Create column was_boosted
omd_table['was_boosted'] = 1

# Select useful columns
omd_table = omd_table[[
    'Organic_ID',
    'was_boosted'
]]

# Add preffix
omd_table = omd_table.add_prefix('omd_')

omd_table

,omd_Organic_ID,omd_was_boosted
0,7511779587924200710,1
1,test,1


In [11]:
# Prepare paid_data for join
# Rename columns
paid_data = paid_data.rename(columns={
    'Spent (USD) in USD (SUM)': 'spend'
})

# Create column was boosted
paid_data['was_boosted'] = (
    paid_data['spend'] > 0
).astype(int)

# Select useful columns
paid_data = paid_data[[
    'Organic_ID',
    'spend',
    'was_boosted'
]]

# Add preffix
paid_data = paid_data.add_prefix('paid_')

paid_data


,paid_Organic_ID,paid_spend,paid_was_boosted
0,7626140510247226645,195.36,1
1,7626140510247226645,189.50,1
2,7626140510247226645,195.70,1
3,7628383524583705877,261.11,1
4,7628383524583705877,236.38,1
...,...,...,...
232,DaV1rEQksi6,34.37,1
233,DaV1rEQksi6,0.71,1
234,DabOq51v5ER,3.47,1
235,DadpNGQmCaR,30.55,1


In [12]:
# Join dataframes

final_boosting = final_boosting.merge(combined_profiles, how='left', left_on='Organic_ID', right_on='prof_Organic_ID')
final_boosting = final_boosting.merge(internal_organic, how='left', left_on='Organic_ID', right_on='int_Organic_ID')
final_boosting = final_boosting.merge(omd_table, how='left', left_on='Organic_ID', right_on='omd_Organic_ID')
final_boosting = final_boosting.merge(paid_data, how='left', left_on='Organic_ID', right_on='paid_Organic_ID')

In [13]:
# Create new columns

# Filter which cells to show in the table
final_boosting['final_filter'] = (
    ((final_boosting['content_type'] == 'Owned') & (final_boosting['Accionable'] == 'Boost')) |
    ((final_boosting['content_type'] == 'Creator') & (final_boosting['prof_can_be_boosted'] == 'Si'))
).astype(int)

# Current date
final_boosting['final_date'] = final_boosting['run_datetime'].max()

# Spark final
final_boosting['final_spark'] = final_boosting['prof_spark_code'].where(
    final_boosting['prof_spark_code'].notna(), 'NA'
)

# Final goal
final_boosting['final_goal'] = final_boosting['prof_goal'].combine_first(final_boosting['int_goal'])

# Final budget
final_boosting['final_budget'] = final_boosting['prof_budget'].combine_first(final_boosting['int_budget'])

# Final end_date
final_boosting['final_end_date'] = final_boosting['prof_end_date'].combine_first(final_boosting['int_end_date'])
final_boosting['final_end_date'] = pd.to_datetime(final_boosting['final_end_date'], format='mixed').dt.date

# Final redirectioning_link
final_boosting['final_redirectioning_link'] = final_boosting['prof_redirectioning_link'].combine_first(
    final_boosting['int_redirectioning_link']
)


# Final status
conditions = [
    (final_boosting['content_type'] == 'Owned') & (final_boosting['Accionable'] == 'Boost') & (final_boosting['final_end_date'] > final_boosting['final_date']),
    (final_boosting['content_type'] == 'Creator') & (final_boosting['prof_can_be_boosted'] == 'Si') & (final_boosting['final_end_date'] > final_boosting['final_date']),
    final_boosting['paid_was_boosted'] == 1,
    final_boosting['omd_was_boosted'] == 1,
    (final_boosting['paid_was_boosted'] == 0) & (final_boosting['final_end_date'] < final_boosting['final_date']),
    (final_boosting['prof_can_be_boosted'] == 'Si') & (final_boosting['omd_was_boosted'] != 1) & (final_boosting['final_end_date'] < final_boosting['final_date']),
]
choices = ['Available', 'Available', 'Promoted', 'Promoted', 'Not Promoted / Expired', 'Not Promoted / Expired']

final_boosting['final_status'] = np.select(conditions, choices, default='Not Promoted / Expired')

In [15]:
# Normalize country names
final_boosting['Country'] = final_boosting['Country'].replace({
    'Salvador': 'El Salvador',
    'Republica Dominicana': 'Dominican Republic',
})

In [16]:
# Save final table
# Open the destination sheets file
sh = gc.open_by_key('1nBC7H5vNsUVQ03W5aJkd-8cBknA_eqAJyeVsuJujc_4')
worksheet = sh.get_worksheet(0)

# Replace old data with new data
set_with_dataframe(worksheet, final_boosting)
print("DataFrame saved successfully!")

DataFrame saved successfully!
